In [ ]:
# ======================================================
# PHẦN 2: HUẤN LUYỆN MODEL & XUẤT FILE (CẬP NHẬT HIỂN THỊ KẾT QUẢ)
# ======================================================
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss
from sklearn.utils import class_weight
import ast
import pickle
import os
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1. Load Data
print("🔄 Đang tải dữ liệu...")
BASE_PATH = '/content/drive/MyDrive/tripfinity/data'
ITEMS_PATH = f'{BASE_PATH}/ai_item_tower_export_20251217_181044.csv'
INTERACTIONS_PATH = f'{BASE_PATH}/generated_user_item_interactions_50k_final.csv'

# Fallback local
if not os.path.exists(INTERACTIONS_PATH):
    INTERACTIONS_PATH = 'generated_user_item_interactions_50k_final.csv'
    ITEMS_PATH = 'ai_item_tower_export_20251217_181044.csv'

items = pd.read_csv(ITEMS_PATH)
interactions = pd.read_csv(INTERACTIONS_PATH)

# Xử lý Item
items['unique_item_id'] = items['item_type'] + "_" + items['item_id'].astype(str)
items['price'] = pd.to_numeric(items['price'], errors='coerce').fillna(items['price'].median())
items['latitude'] = items['latitude'].fillna(items['latitude'].mean())
items['longitude'] = items['longitude'].fillna(items['longitude'].mean())

def clean_features(x):
    try:
        if isinstance(x, str) and "[" in x:
            return " ".join(ast.literal_eval(x)).replace("_", " ")
        return "general"
    except:
        return "general"
items['feature_text'] = items['normalized_features'].apply(clean_features)

🔄 Đang tải dữ liệu...


In [ ]:
# 2. Feature Engineering
print("⏳ Đang xử lý User Profiles...")
merged = interactions.merge(items, on='unique_item_id')
user_profiles = merged.groupby('user_id').agg({
    'price': lambda x: np.mean(np.log1p(x)),
    'latitude': 'mean',
    'longitude': 'mean',
    'feature_text': lambda x: " ".join(x)
}).rename(columns={'price': 'u_price', 'latitude': 'u_lat', 'longitude': 'u_lon', 'feature_text': 'u_text'}).reset_index()

⏳ Đang xử lý User Profiles...


In [ ]:
# 3. Tạo Dataset
print("⚡ Đang tạo Dataset...")
positives = interactions[['user_id', 'unique_item_id']].copy()
positives['label'] = 1

n_neg = len(positives) * 3
neg_u = np.random.choice(user_profiles['user_id'].values, n_neg)
neg_i = np.random.choice(items['unique_item_id'].values, n_neg)
negatives = pd.DataFrame({'user_id': neg_u, 'unique_item_id': neg_i, 'label': 0})

dataset = pd.concat([positives, negatives], ignore_index=True)
dataset = dataset.merge(user_profiles, on='user_id', how='inner')
dataset = dataset.merge(items[['unique_item_id', 'latitude', 'longitude', 'price', 'item_type', 'feature_text']], on='unique_item_id')
dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)

# 4. Scalers & Encoders
print("⚡ Đang chuẩn hóa...")
scaler_lat = MinMaxScaler().fit(items[['latitude']].values)
scaler_lon = MinMaxScaler().fit(items[['longitude']].values)
scaler_price = MinMaxScaler().fit(np.vstack([np.log1p(items[['price']].values), dataset[['u_price']].values]))

type_encoder = LabelEncoder()
items['type_encoded'] = type_encoder.fit_transform(items['item_type'])
dataset['type_encoded'] = type_encoder.transform(dataset['item_type'])

MAX_TOKENS = 5000
text_vectorizer = layers.TextVectorization(max_tokens=MAX_TOKENS, output_sequence_length=30)
text_vectorizer.adapt(np.concatenate([items['feature_text'].values, user_profiles['u_text'].values]))

def get_inputs(df):
    return {
        "user_lat": scaler_lat.transform(df[['u_lat']].values),
        "user_lon": scaler_lon.transform(df[['u_lon']].values),
        "user_price": scaler_price.transform(df[['u_price']].values),
        "user_text": df['u_text'].values,
        "item_lat": scaler_lat.transform(df[['latitude']].values),
        "item_lon": scaler_lon.transform(df[['longitude']].values),
        "item_price": scaler_price.transform(np.log1p(df[['price']].values)),
        "item_type": df['type_encoded'].values,
        "item_text": df['feature_text'].values
    }

⚡ Đang tạo Dataset...
⚡ Đang chuẩn hóa...


In [ ]:
# 5. Chia tập Train/Test (Để đánh giá sơ bộ)
X_full = get_inputs(dataset)
y_full = dataset['label'].values
indices = np.arange(len(y_full))
train_idx, test_idx, y_train, y_test = train_test_split(indices, y_full, test_size=0.2, random_state=42, stratify=y_full)

def subset_dict(d, idx): return {k: v[idx] for k, v in d.items()}
train_inputs = subset_dict(X_full, train_idx)
test_inputs = subset_dict(X_full, test_idx)

In [ ]:
# 6. Build Model
def create_model():
    # User Tower
    u_lat = layers.Input(shape=(1,), name='user_lat')
    u_lon = layers.Input(shape=(1,), name='user_lon')
    u_price = layers.Input(shape=(1,), name='user_price')
    u_text = layers.Input(shape=(1,), dtype=tf.string, name='user_text')

    u_txt_emb = layers.GlobalAveragePooling1D()(layers.Embedding(MAX_TOKENS, 32)(text_vectorizer(u_text)))
    u_vec = layers.Concatenate()([u_lat, u_lon, u_price, u_txt_emb])
    u_vec = layers.Dense(128, activation='relu')(u_vec)
    u_vec = layers.BatchNormalization()(u_vec)
    u_vec = layers.Dropout(0.3)(u_vec)
    u_vec = layers.Dense(64, activation='relu')(u_vec)
    u_out = layers.Dense(32, name='u_embedding')(u_vec)

    # Item Tower
    i_lat = layers.Input(shape=(1,), name='item_lat')
    i_lon = layers.Input(shape=(1,), name='item_lon')
    i_price = layers.Input(shape=(1,), name='item_price')
    i_type = layers.Input(shape=(1,), name='item_type')
    i_text = layers.Input(shape=(1,), dtype=tf.string, name='item_text')

    i_type_vec = layers.Flatten()(layers.Embedding(10, 8)(i_type))
    i_txt_emb = layers.GlobalAveragePooling1D()(layers.Embedding(MAX_TOKENS, 32)(text_vectorizer(i_text)))
    i_vec = layers.Concatenate()([i_lat, i_lon, i_price, i_type_vec, i_txt_emb])
    i_vec = layers.Dense(128, activation='relu')(i_vec)
    i_vec = layers.BatchNormalization()(i_vec)
    i_vec = layers.Dropout(0.3)(i_vec)
    i_vec = layers.Dense(64, activation='relu')(i_vec)
    i_out = layers.Dense(32, name='i_embedding')(i_vec)

    out = layers.Dense(1, activation='sigmoid')(layers.Dot(axes=1, normalize=True)([u_out, i_out]))
    model = keras.Model(inputs=[u_lat, u_lon, u_price, u_text, i_lat, i_lon, i_price, i_type, i_text], outputs=out)
    model.compile(optimizer=keras.optimizers.Adam(0.001), loss='binary_crossentropy', metrics=['accuracy', 'AUC'])
    return model

model = create_model()

In [ ]:
# ======================================================
# 7. Train & Đánh giá (Split Data) - CÓ REPORT
# ======================================================
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

print("\n🚀 GIAI ĐOẠN 1: TRAIN & ĐÁNH GIÁ (80% DATA)...")
cw = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)

# Train model
history = model.fit(
    train_inputs, y_train,
    validation_data=(test_inputs, y_test),
    epochs=20,
    batch_size=256,
    class_weight=dict(enumerate(cw)),
    verbose=1
)

# --- TÍNH TOÁN CHỈ SỐ REPORT ---
y_pred_test = model.predict(test_inputs, verbose=0)

# 1. Accuracy & AUC
acc_test = accuracy_score(y_test, (y_pred_test > 0.5).astype(int))
auc_test = roc_auc_score(y_test, y_pred_test)

# 2. Log Loss (Hàm mất mát trên tập test)
test_loss = log_loss(y_test, y_pred_test)

# 3. Tính Overfitting Score
# Công thức: (Loss tập Val - Loss tập Train) ở Epoch cuối cùng
# Nếu số này càng lớn > 0 -> Overfit. Nếu gần 0 -> Tốt. Nếu < 0 -> Underfit/Tốt.
final_train_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]
overfit_score = final_val_loss - final_train_loss

# --- IN REPORT ĐẸP ---
print("\n" + "="*30)
print("📊 KẾT QUẢ CUỐI CÙNG (FINAL REPORT)")
print("="*30)
print(f"✅ Accuracy:    {acc_test:.4f}")
print(f"✅ AUC Score:   {auc_test:.4f}")
print(f"✅ Log Loss:    {test_loss:.4f}")
print(f"⚠️ Overfitting: {overfit_score:.4f} (Val Loss - Train Loss)")
print("="*30 + "\n")

# Logic cảnh báo nhanh
if overfit_score > 0.05:
    print("⚠️ CẢNH BÁO: Model có dấu hiệu Overfitting nhẹ (Train tốt hơn Test).")
elif overfit_score < 0:
    print("🎉 TỐT: Model tổng quát hóa rất tốt (Val Loss thấp hơn cả Train Loss).")
else:
    print("✅ TỐT: Model ổn định (Train và Val tương đương).")


🚀 GIAI ĐOẠN 1: TRAIN & ĐÁNH GIÁ (80% DATA)...
Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 24s 38ms/step - AUC: 0.8638 - accuracy: 0.7477 - loss: 0.4558 - val_AUC: 0.8628 - val_accuracy: 0.7370 - val_loss: 0.4670
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 31ms/step - AUC: 0.8613 - accuracy: 0.7423 - loss: 0.4596 - val_AUC: 0.8622 - val_accuracy: 0.7521 - val_loss: 0.4548
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - AUC: 0.8632 - accuracy: 0.7464 - loss: 0.4560 - val_AUC: 0.8625 - val_accuracy: 0.7457 - val_loss: 0.4611
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 42s 35ms/step - AUC: 0.8663 - accuracy: 0.7506 - loss: 0.4519 - val_AUC: 0.8627 - val_accuracy: 0.7409 - val_loss: 0.4700
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 31ms/step - AUC: 0.8653 - accuracy: 0.7458 - loss: 0.4525 - val_AUC: 0.8618 - val_accuracy: 0.7364 - val_loss: 0.4734
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 22s 36ms/step - AUC: 0.8656 - accuracy: 0.7481 - loss: 0.4511 - val_AUC: 0.8614 - val_accuracy: 0.7499 

In [ ]:
# 8. TRAIN FULL (100% DATA)
print("\n🚀 GIAI ĐOẠN 2: RETRAIN TRÊN 100% DỮ LIỆU (FULL DATASET)...")
cw_full = class_weight.compute_class_weight('balanced', classes=np.unique(y_full), y=y_full)

# Train tiếp model hiện tại với toàn bộ dữ liệu
model.fit(
    X_full, y_full,
    epochs=15,
    batch_size=512,
    class_weight=dict(enumerate(cw_full)),
    verbose=1
)

# --- THÊM PHẦN ĐÁNH GIÁ CUỐI CÙNG (THEO YÊU CẦU CỦA BẠN) ---
print("\n🔄 Đang tính toán chỉ số trên toàn bộ tập dữ liệu...")
y_pred_full = model.predict(X_full, verbose=0)
acc_full = accuracy_score(y_full, (y_pred_full > 0.5).astype(int))
auc_full = roc_auc_score(y_full, y_pred_full)
loss_full = log_loss(y_full, y_pred_full)

print("\n" + "="*40)
print("📊 KẾT QUẢ CUỐI CÙNG (SAU KHI TRAIN 100% DATA)")
print("="*40)
print(f"✅ Final Training Accuracy: {acc_full:.4f} (Độ thuộc bài)")
print(f"✅ Final Training AUC:      {auc_full:.4f}")
print(f"✅ Final Training Loss:     {loss_full:.4f}")
print("="*40)


🚀 GIAI ĐOẠN 2: RETRAIN TRÊN 100% DỮ LIỆU (FULL DATASET)...
Epoch 1/15
391/391 ━━━━━━━━━━━━━━━━━━━━ 21s 48ms/step - AUC: 0.8679 - accuracy: 0.7516 - loss: 0.4485
Epoch 2/15
391/391 ━━━━━━━━━━━━━━━━━━━━ 17s 43ms/step - AUC: 0.8684 - accuracy: 0.7482 - loss: 0.4479
Epoch 3/15
391/391 ━━━━━━━━━━━━━━━━━━━━ 17s 44ms/step - AUC: 0.8690 - accuracy: 0.7501 - loss: 0.4476
Epoch 4/15
391/391 ━━━━━━━━━━━━━━━━━━━━ 17s 44ms/step - AUC: 0.8674 - accuracy: 0.7518 - loss: 0.4490
Epoch 5/15
391/391 ━━━━━━━━━━━━━━━━━━━━ 19s 48ms/step - AUC: 0.8682 - accuracy: 0.7506 - loss: 0.4492
Epoch 6/15
391/391 ━━━━━━━━━━━━━━━━━━━━ 17s 44ms/step - AUC: 0.8686 - accuracy: 0.7517 - loss: 0.4475
Epoch 7/15
391/391 ━━━━━━━━━━━━━━━━━━━━ 17s 44ms/step - AUC: 0.8675 - accuracy: 0.7490 - loss: 0.4484
Epoch 8/15
391/391 ━━━━━━━━━━━━━━━━━━━━ 17s 44ms/step - AUC: 0.8684 - accuracy: 0.7514 - loss: 0.4490
Epoch 9/15
391/391 ━━━━━━━━━━━━━━━━━━━━ 21s 45ms/step - AUC: 0.8668 - accuracy: 0.7481 - loss: 0.4494
Epoch 10/15
391/391 ━━

In [ ]:
# ======================================================
# 9. XUẤT FILE (LƯU 2 FILE VÀO DRIVE)
# ======================================================
import os
import pickle

# Định nghĩa đường dẫn lưu vào Drive (Dùng BASE_PATH)
model_path = f'{BASE_PATH}/tripfinity_recsys_model.keras'
artifacts_path = f'{BASE_PATH}/recsys_artifacts.pkl'

print("\n💾 Đang xuất file...")

# 1. Lưu Model (File .keras) -> Lưu cấu trúc mạng & trọng số
model.save(model_path)
print(f"   -> Đã lưu Model tại: {model_path}")

# 2. Lưu Artifacts (File .pkl) -> Lưu Scaler, Encoder, Vocab
artifacts = {
    'scaler_lat': scaler_lat,
    'scaler_lon': scaler_lon,
    'scaler_price': scaler_price,
    'type_encoder': type_encoder,
    'vocab': text_vectorizer.get_vocabulary(),
    'vectorizer_config': text_vectorizer.get_config(),
    'user_profiles_dict': user_profiles.set_index('user_id').to_dict('index')
}

with open(artifacts_path, 'wb') as f:
    pickle.dump(artifacts, f)
print(f"   -> Đã lưu Artifacts tại: {artifacts_path}")

print("\n✅ ĐÃ HOÀN TẤT! Cả 2 file đã nằm an toàn trong Drive.")


💾 Đang xuất file...
   -> Đã lưu Model tại: /content/drive/MyDrive/tripfinity/data/tripfinity_recsys_model.keras
   -> Đã lưu Artifacts tại: /content/drive/MyDrive/tripfinity/data/recsys_artifacts.pkl

✅ ĐÃ HOÀN TẤT! Cả 2 file đã nằm an toàn trong Drive.


In [ ]:
# ======================================================
# PHẦN 3: GIAO DIỆN TEST MODEL (SIMPLE ID INPUT)
# ======================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pickle
import os

# --- 1. CẤU HÌNH MẶC ĐỊNH ---
# Địa điểm mặc định cho User mới (Đà Nẵng)
DEFAULT_LAT = 16.0544
DEFAULT_LON = 108.2022
RADIUS_KM = 15

# --- 2. LOAD DATA & MODEL ---
print("🔄 Đang khởi động hệ thống gợi ý...")
BASE_PATH = '/content/drive/MyDrive/tripfinity/data'
ITEMS_PATH = f'{BASE_PATH}/ai_item_tower_export_20251217_181044.csv'
if not os.path.exists(ITEMS_PATH): ITEMS_PATH = 'ai_item_tower_export_20251217_181044.csv'

items = pd.read_csv(ITEMS_PATH)
# Preprocessing
items['unique_item_id'] = items['item_type'] + "_" + items['item_id'].astype(str)
items['price'] = pd.to_numeric(items['price'], errors='coerce').fillna(items['price'].median())
items['feature_text'] = items['normalized_features'].astype(str).str.replace(r'[\[\]"]', '', regex=True).str.replace(',', ' ')

# Load Model & Artifacts
with open(f'{BASE_PATH}/recsys_artifacts.pkl', 'rb') as f:
    artifacts = pickle.load(f)
model = keras.models.load_model(f'{BASE_PATH}/tripfinity_recsys_model.keras')

# Setup Scalers
sc_lat = artifacts['scaler_lat']
sc_lon = artifacts['scaler_lon']
sc_price = artifacts['scaler_price']
enc_type = artifacts['type_encoder']
u_profiles = artifacts['user_profiles_dict']

# Hàm tính khoảng cách
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

# --- 3. LOGIC GỢI Ý (THÔNG MINH) ---
def get_smart_recommendations(user_id):
    # Biến để xác định tâm tìm kiếm
    target_lat, target_lon = 0, 0

    # A. XỬ LÝ USER INPUT
    if user_id in u_profiles:
        # === TRƯỜNG HỢP 1: USER CŨ (Đã có lịch sử) ===
        prof = u_profiles[user_id]

        # Lấy sở thích đã học được
        u_lat_val, u_lon_val = prof['u_lat'], prof['u_lon']
        u_price_val = prof['u_price'] # Đã log
        u_text_val = str(prof['u_text'])

        # Tâm tìm kiếm chính là sở thích của họ
        target_lat, target_lon = u_lat_val, u_lon_val
        status = "✅ USER CŨ (Welcome Back!)"
        desc = f"Hệ thống nhớ rằng bạn thích khu vực ({u_lat_val:.2f}, {u_lon_val:.2f}) và các dịch vụ tương tự lịch sử cũ."

    else:
        # === TRƯỜNG HỢP 2: USER MỚI (Cold Start) ===
        # Mặc định gán về Đà Nẵng
        u_lat_val, u_lon_val = DEFAULT_LAT, DEFAULT_LON
        # Giá mặc định trung bình
        u_price_val = np.log1p(items['price'].median())
        u_text_val = "general"

        # Tâm tìm kiếm là Đà Nẵng
        target_lat, target_lon = DEFAULT_LAT, DEFAULT_LON
        status = "✨ USER MỚI (Newbie)"
        desc = f"Chưa có dữ liệu. Hệ thống mặc định gợi ý các dịch vụ Hot tại ĐÀ NẴNG."

    # B. CHUẨN BỊ INPUT CHO MODEL
    N = len(items)

    # Input User (Broadcast N lần)
    u_lat_in = sc_lat.transform([[u_lat_val]])[0][0]
    u_lon_in = sc_lon.transform([[u_lon_val]])[0][0]
    u_price_in = sc_price.transform([[u_price_val]])[0][0]

    # Input Item
    i_price_log = np.log1p(items['price'].values).reshape(-1, 1)

    inputs = {
        "user_lat": np.full((N, 1), u_lat_in),
        "user_lon": np.full((N, 1), u_lon_in),
        "user_price": np.full((N, 1), u_price_in),
        "user_text": tf.constant([u_text_val] * N, dtype=tf.string),
        "item_lat": sc_lat.transform(items[['latitude']].values),
        "item_lon": sc_lon.transform(items[['longitude']].values),
        "item_price": sc_price.transform(i_price_log),
        "item_type": enc_type.transform(items['item_type']),
        "item_text": tf.constant(items['feature_text'].values.astype(str), dtype=tf.string)
    }

    # C. DỰ ĐOÁN & XẾP HẠNG
    scores = model.predict(inputs, verbose=0).flatten()

    res = items.copy()
    res['score'] = scores
    # Tính khoảng cách từ "Tâm tìm kiếm" (Sở thích cũ HOẶC Đà Nẵng)
    res['dist_km'] = haversine(target_lat, target_lon, res['latitude'], res['longitude'])

    # Logic hiển thị: Lấy Top trong bán kính trước, nếu thiếu lấy thêm ở xa
    in_zone = res[res['dist_km'] <= RADIUS_KM].sort_values('score', ascending=False)
    out_zone = res[res['dist_km'] > RADIUS_KM].sort_values('score', ascending=False)

    final = pd.concat([in_zone.head(10), out_zone.head(5)])
    final['price_fmt'] = final['price'].apply(lambda x: f"{int(x):,} đ")

    return final[['title', 'item_type', 'price_fmt', 'dist_km', 'score', 'location']], status, desc

# --- 4. GIAO DIỆN (UI) ---
style = {'description_width': 'initial'}
txt_uid = widgets.IntText(value=0, description='👉 NHẬP USER ID:', style={'description_width': 'initial', 'font_weight': 'bold'})
btn = widgets.Button(description='Xem Gợi Ý', button_style='primary')
out = widgets.Output()

def on_click(b):
    with out:
        clear_output()
        uid = txt_uid.value
        if uid <= 0:
            print("❌ Vui lòng nhập User ID lớn hơn 0")
            return

        print(f"🔎 Đang tải profile cho User {uid}...")
        try:
            df_res, status, desc = get_smart_recommendations(uid)

            # Hiển thị thông tin trạng thái đẹp mắt
            print("="*60)
            print(f"{status}")
            print(f"ℹ️ {desc}")
            print("="*60)

            display(df_res)
        except Exception as e:
            print(f"❌ Lỗi: {e}")

btn.on_click(on_click)

print("\n📱 MÀN HÌNH HOME (MÔ PHỎNG)")
display(widgets.HBox([txt_uid, btn]))
display(out)

🔄 Đang khởi động hệ thống gợi ý...

📱 MÀN HÌNH HOME (MÔ PHỎNG)


Output()